## Singly Linked List

In [1]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None    # None = "nothing after this" (like Java's null)


# building 1 -> 2 -> 3 -> None
head = Node(1)
head.next = Node(2)
head.next.next = Node(3)
# head.next.next.next is left as None (the default) — this IS the end

current = head
while current is not None:
    print(current.value)
    current = current.next

1
2
3


A `Node` just bundles a value with a `next` pointer - `self` here is what `this` is in Java, and `None` is Python's `null`. `head` is the only thing the list "remembers"; every other node is only reachable by following `.next` pointers from it. Walking the chain with `while current is not None` is the pattern to remember - it stops the moment you fall off the end.

## Doubly Linked List

In [2]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None
        self.prev = None


# building 1 <-> 2 <-> 3
head = Node(1)
middle = Node(2)
tail = Node(3)

head.next = middle
middle.prev = head
middle.next = tail
tail.prev = middle

# walk forward
current = head
while current is not None:
    print(current.value)
    current = current.next

# walk backward, starting from tail
current = tail
while current is not None:
    print(current.value)
    current = current.prev

1
2
3
3
2
1


Same idea, but every node also remembers `.prev`, so you can walk backward as easily as forward - that's the whole difference from a singly linked list. The cost is bookkeeping: every insert/removal now has two pointers to update per side instead of one. (Side note: Java's own `java.util.LinkedList` is actually implemented as a doubly linked list under the hood, not singly.)

## Circular Linked List

In [3]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None


# building a circular list: 1 -> 2 -> 3 -> back to 1
head = Node(1)
head.next = Node(2)
head.next.next = Node(3)
head.next.next.next = head    # <- this is the whole trick: tail points back to head

The one-line trick that makes it circular: `head.next.next.next = head` instead of leaving the last node's `.next` as `None`. This cell only builds the loop, it doesn't walk it - and that's on purpose, because `while current is not None` would never stop here (there's no `None` anywhere in the chain). Traversing this safely needs a different stopping condition, like `while True: ... ; if current is head: break`.

## Insert Into a Linked List

In [4]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None


def insert_at(head, position, value):
    """Insert `value` so it ends up at `position` (1 = first place)."""
    new_node = Node(value)

    # Special case: inserting at the very front.
    # There's no "previous node" to rewire - the new node just BECOMES the head.
    if position == 1:
        new_node.next = head
        return new_node

    # Otherwise, walk to the node sitting just BEFORE the target spot.
    current = head
    for _ in range(position - 2):
        current = current.next

    # Slot the new node in between current and whatever came after it.
    new_node.next = current.next
    current.next = new_node

    return head


def print_list(head):
    values = []
    current = head
    while current is not None:
        values.append(str(current.value))
        current = current.next
    print(" -> ".join(values))

head = Node(10)
head.next = Node(20)
head.next.next = Node(30)

print("before:")
print_list(head)

head = insert_at(head, 2, 99)

print("after:")
print_list(head)

before:
10 -> 20 -> 30
after:
10 -> 99 -> 20 -> 30


Two details worth remembering here: the loop walks `position - 2` steps (not `position - 1`) because it needs to stop at the node just *before* the gap, and position 1 needs special-casing since there's no "previous node" to rewire when inserting at the very front - the new node just becomes the head, and the caller has to store that returned value (`head = insert_at(head, ...)`), since the function can't reach back and change the caller's variable on its own.

## Delete From a Linked List

The natural follow-up to `insert_at`: remove the node sitting at a given position instead of adding one. Same "walk to the node before the target" idea, just rewiring around a node instead of into a gap. Reuses the `Node` class and `print_list` helper already defined above.

In [5]:
def delete_at(head, position):
    """Delete the node sitting at `position` (1 = first place) and return the new head."""
    if head is None:
        return head

    # Special case: deleting the head itself - no "previous node" to rewire,
    # the SECOND node just becomes the new head.
    if position == 1:
        return head.next

    # Otherwise, walk to the node sitting just BEFORE the one we want to delete
    # (same "- 2" logic as insert_at, for the same reason).
    current = head
    for _ in range(position - 2):
        current = current.next

    # Skip over the target node entirely by pointing current.next PAST it.
    current.next = current.next.next

    return head


head2 = Node(10)
head2.next = Node(20)
head2.next.next = Node(30)
head2.next.next.next = Node(40)

print("before:")
print_list(head2)

head2 = delete_at(head2, 2)
print("after deleting position 2:")
print_list(head2)

head2 = delete_at(head2, 1)
print("after deleting position 1 (the head):")
print_list(head2)

before:
10 -> 20 -> 30 -> 40
after deleting position 2:
10 -> 30 -> 40
after deleting position 1 (the head):
30 -> 40


`current.next = current.next.next` is the whole trick - `current` is the node right before the deleted one, and this line just makes it point two steps ahead instead of one, skipping the doomed node entirely. Nothing has to explicitly "destroy" that node - once nothing points to it anymore, Python's garbage collector cleans it up automatically (no manual `free()` needed, unlike C).